# 01 — Dataset Curation

Construir el dataset de code review en español para fine-tuning.

**Formato objetivo:** JSONL con campos `instruction`, `input`, `output`

**Target:** 300-500 ejemplos de calidad.

In [ ]:
# Instalar dependencias (solo primera vez)
# !pip install datasets pandas tqdm

In [ ]:
import json
import os
import random
from pathlib import Path

import pandas as pd

random.seed(42)
DATA_DIR = Path('../data')
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_PATH = DATA_DIR / 'dataset.jsonl'

## 1. Seed examples — errores comunes en Python

In [ ]:
# Ejemplos semilla — ampliar hasta 300-500
SEED_EXAMPLES = [
    {
        "instruction": "Revisa este código Python y da feedback detallado:",
        "input": "def suma(a, b):\n  return a - b",
        "output": "Error lógico: la función está restando en lugar de sumar. Corrige `return a - b` → `return a + b`. Además, por convención PEP8 usa 4 espacios de indentación."
    },
    {
        "instruction": "¿Qué problema tiene este código?",
        "input": "for i in range(10):\nprint(i)",
        "output": "Error de indentación: `print(i)` debe estar indentado con 4 espacios dentro del bloque `for`. Corrección:\n```python\nfor i in range(10):\n    print(i)\n```"
    },
    {
        "instruction": "Revisa este código Python y da feedback detallado:",
        "input": "lista = [1, 2, 3]\nprint(lista[5])",
        "output": "Error en tiempo de ejecución: `IndexError: list index out of range`. La lista tiene 3 elementos (índices 0, 1, 2) pero se accede al índice 5. Verifica siempre los límites con `len(lista) - 1` o usa `try/except IndexError`."
    },
    {
        "instruction": "¿Qué problema tiene este código?",
        "input": "def divide(a, b):\n    return a / b\n\nresultado = divide(10, 0)",
        "output": "Error en tiempo de ejecución: `ZeroDivisionError`. La función no valida que `b != 0`. Agrega una guarda:\n```python\ndef divide(a, b):\n    if b == 0:\n        raise ValueError('El divisor no puede ser cero')\n    return a / b\n```"
    },
    {
        "instruction": "Revisa este código Python y da feedback detallado:",
        "input": "def calcular_promedio(numeros):\n    total = 0\n    for n in numeros:\n        total = total + n\n    promedio = total / len(numeros)\n    return promedio",
        "output": "El código es correcto pero tiene dos mejoras posibles:\n1. Usar `sum()` en lugar del bucle: `total = sum(numeros)`\n2. No maneja lista vacía (`ZeroDivisionError`). Agrega: `if not numeros: return 0`\n\nVersión mejorada:\n```python\ndef calcular_promedio(numeros):\n    if not numeros:\n        return 0\n    return sum(numeros) / len(numeros)\n```"
    }
]

print(f"Ejemplos semilla: {len(SEED_EXAMPLES)}")

## 2. Validación del formato

In [ ]:
def validate_sample(sample: dict) -> bool:
    required = ['instruction', 'input', 'output']
    for field in required:
        if field not in sample:
            return False
        if not isinstance(sample[field], str):
            return False
    if len(sample['output'].strip()) < 20:
        return False
    return True

valid = [s for s in SEED_EXAMPLES if validate_sample(s)]
invalid = [s for s in SEED_EXAMPLES if not validate_sample(s)]
print(f"Válidos: {len(valid)} | Inválidos: {len(invalid)}")

## 3. Guardar como JSONL

In [ ]:
def save_jsonl(data: list[dict], path: Path) -> None:
    with open(path, 'w', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

save_jsonl(valid, OUTPUT_PATH)
print(f"Guardado: {OUTPUT_PATH} ({len(valid)} ejemplos)")

## 4. Estadísticas

In [ ]:
df = pd.DataFrame(valid)
df['input_len'] = df['input'].str.len()
df['output_len'] = df['output'].str.len()

print("=== Estadísticas del dataset ===")
print(f"Total ejemplos: {len(df)}")
print(f"Input avg length: {df['input_len'].mean():.0f} chars")
print(f"Output avg length: {df['output_len'].mean():.0f} chars")
df[['input_len', 'output_len']].describe()

## Próximos pasos

- [ ] Ampliar hasta 300-500 ejemplos manualmente o con ayuda de ChatGPT/Claude
- [ ] Cubrir más tipos de errores: tipos, recursión, imports, POO
- [ ] Revisar calidad de los outputs (human review)
- [ ] Continuar con `02_finetune_qlora.ipynb`